# 12 · Corner Case Mining：从场景属性到失败切片

安全工作不只是写一个 fallback rule，还要知道失败发生在什么分布切片：雨天、遮挡、传感器 dropout、时间延迟、交通密度或组合条件。本 notebook 生成一批带属性的 episode，模拟一个模型风险分数，并做 slice-level failure analysis。

学习目标：

- 把 scenario attributes 和 episode outcome 放进同一张表；
- 计算整体指标与 slice 指标，发现平均数掩盖的风险；
- 用 model score 选择 hard negative / replay 候选；
- 讨论数据挖掘中的 leakage、selection bias 和 calibration。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

rng = np.random.default_rng(44)
weather_values = ['clear', 'rain', 'fog']
density_values = ['low', 'medium', 'high']

def make_scenarios(n=1200, seed=44):
    local = np.random.default_rng(seed)
    weather = local.choice(weather_values, n, p=[0.55, 0.3, 0.15])
    density = local.choice(density_values, n, p=[0.35, 0.45, 0.20])
    dropout = local.random(n) < 0.14
    occlusion = local.random(n) < 0.18
    latency_ms = local.gamma(shape=2.0, scale=18.0, size=n)
    weather_risk = np.select([weather == 'rain', weather == 'fog'], [0.16, 0.28], default=0.0)
    density_risk = np.select([density == 'medium', density == 'high'], [0.10, 0.22], default=0.0)
    latent_risk = (
        0.08 + weather_risk + density_risk
        + 0.18 * dropout + 0.20 * occlusion
        + 0.004 * latency_ms
        + local.normal(0, 0.08, n)
    )
    event = latent_risk > 0.48
    model_score = np.clip(latent_risk + local.normal(0, 0.13, n), 0, 1)
    return pd.DataFrame({
        'episode_id': np.arange(n),
        'weather': weather,
        'density': density,
        'sensor_dropout': dropout,
        'occlusion': occlusion,
        'latency_ms': latency_ms,
        'latent_risk': np.clip(latent_risk, 0, 1),
        'failure_event': event,
        'model_score': model_score,
    })

episodes = make_scenarios()
print('overall failure rate:', episodes['failure_event'].mean())


In [ ]:
def slice_report(df, column):
    report = df.groupby(column, observed=True).agg(
        episodes=('episode_id', 'count'),
        failure_rate=('failure_event', 'mean'),
        mean_score=('model_score', 'mean'),
        latency_p95=('latency_ms', lambda x: np.percentile(x, 95)),
    ).sort_values('failure_rate', ascending=False)
    return report

print('weather slices')
print(slice_report(episodes, 'weather').round(3).to_string())
print('\ndensity slices')
print(slice_report(episodes, 'density').round(3).to_string())


In [ ]:
def threshold_metrics(df, threshold):
    pred = df['model_score'] >= threshold
    truth = df['failure_event']
    tp = np.sum(pred & truth)
    fp = np.sum(pred & ~truth)
    fn = np.sum(~pred & truth)
    recall = tp / max(tp + fn, 1)
    precision = tp / max(tp + fp, 1)
    return precision, recall

def show_mining(threshold=0.55, top_k=80, latency_cut=70.0):
    precision, recall = threshold_metrics(episodes, threshold)
    selected = episodes.sort_values('model_score', ascending=False).head(top_k)
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    weather_report = slice_report(episodes, 'weather')
    weather_report['failure_rate'].plot(kind='bar', ax=ax[0], color='tab:orange')
    ax[0].set_ylim(0, 1)
    ax[0].set_title('failure rate by weather')
    ax[0].set_ylabel('episode failure rate')
    ax[1].scatter(episodes['latency_ms'], episodes['model_score'], s=8, alpha=0.25)
    ax[1].scatter(selected['latency_ms'], selected['model_score'], s=16, c='tab:red', label='top replay candidates')
    ax[1].axvline(latency_cut, color='black', linestyle='--', label='latency cut')
    ax[1].set_xlabel('latency / ms')
    ax[1].set_ylabel('model risk score')
    ax[1].set_title(f'precision={precision:.3f}, recall={recall:.3f}')
    ax[1].legend()
    plt.tight_layout()
    plt.show()
    print('selected slice counts:')
    print(selected[['weather', 'density', 'sensor_dropout', 'occlusion']].value_counts().head(10))

interact(
    show_mining,
    threshold=FloatSlider(min=0.1, max=0.9, step=0.05, value=0.55, description='score threshold'),
    top_k=IntSlider(min=20, max=300, step=20, value=80, description='replay top K'),
    latency_cut=FloatSlider(min=0.0, max=160.0, step=10.0, value=70.0, description='latency cut'),
);


### 练习：切片不是免费午餐

- 找出一个 failure rate 高但样本数很少的 slice，讨论置信区间和是否需要补采样。
- 比较按 model score 排名与按 latency / occlusion 规则筛选的 replay 集。
- 随机把场景属性泄漏进训练/测试划分，观察 slice 指标为什么会虚高。
- 为每个 hard case 保存原始传感器、模型版本、规则 reason code 和可复现 seed。


In [ ]:
thresholds = np.linspace(0.1, 0.9, 17)
precision, recall = zip(*(threshold_metrics(episodes, value) for value in thresholds))
plt.plot(thresholds, precision, label='precision')
plt.plot(thresholds, recall, label='recall')
plt.xlabel('risk threshold')
plt.ylabel('metric')
plt.title('hard-case detector operating point')
plt.legend()
plt.show()


## 完成标准

- 交付一个 slice report，至少包含 weather、density、dropout、occlusion 和 latency。
- 选择一个 replay policy，并报告 precision、recall、样本数和潜在偏差。
- 找到一个平均指标正常、但 slice 指标明显失败的 case。
- 说明如何把这个 notebook 迁移到真实日志、CARLA、SafeBench 或 scenario database。
